# Notebook 02 — Semantic Grading Models

Compares the three grading models used in the ablation study:

| Model | Components |
|-------|------------|
| **A — ML** | TF-IDF bigrams + Jaccard keyword overlap |
| **B — DL** | RoBERTa cross-encoder + SBERT bi-encoder |
| **C — Hybrid** | Cross-encoder + BiLSTM quality scorer + TF-IDF |

## Key Concepts
- **Bi-encoder**: Encodes student and teacher separately; fast but misses cross-text interactions.
- **Cross-encoder**: Encodes both texts jointly via `[CLS] student [SEP] teacher [SEP]`;
  full cross-attention = more accurate, but non-cacheable.
- **BiLSTM + Attention**: Sentence-level structural quality; trained with Xavier/Orthogonal init.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'engine'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, CrossEncoder, util

print('Libraries loaded.')

## 1. Test Dataset

In [ ]:
# (teacher_answer, student_answer, expected_label)
# expected_label: 'high'=>0.7, 'medium'=0.4-0.7, 'low'=<0.4
SAMPLES = [
    (
        'Photosynthesis is the process by which green plants use sunlight, water, and carbon dioxide to produce glucose and oxygen.',
        'Plants use sunlight and CO2 to make sugar and release O2 through photosynthesis.',
        'high'
    ),
    (
        'Newton second law of motion states that force equals mass times acceleration, F = ma.',
        'Force is how hard you push something. Heavier things need more force to move.',
        'medium'
    ),
    (
        'The mitochondria is the powerhouse of the cell, responsible for producing ATP through cellular respiration.',
        'The water cycle involves evaporation, condensation, and precipitation.',
        'low'
    ),
    (
        'Osmosis is the movement of water molecules from a region of lower solute concentration to higher solute concentration through a semi-permeable membrane.',
        'Water moves from dilute to concentrated solution across a membrane by osmosis.',
        'high'
    ),
    (
        'The French Revolution began in 1789 and ended the monarchy, establishing the principles of liberty, equality, and fraternity.',
        'The revolution in France happened in the late 1700s and changed the government system.',
        'medium'
    ),
]

teachers = [s[0] for s in SAMPLES]
students = [s[1] for s in SAMPLES]
labels   = [s[2] for s in SAMPLES]

print(f'Dataset: {len(SAMPLES)} answer pairs')
for i, (t, s, l) in enumerate(SAMPLES):
    print(f'\n[{i+1}] Expected: {l}')
    print(f'  Teacher: {t[:70]}...')
    print(f'  Student: {s[:70]}...')

## 2. Model A — TF-IDF + Jaccard (ML Baseline)

In [ ]:
from answer_scorer import ml_similarity, _tfidf_cosine, _jaccard_keyword

print('TF-IDF + Jaccard scores (Model A — ML Baseline)')
print('='*60)

ml_scores = []
for i, (t, s, l) in enumerate(SAMPLES):
    tfidf = _tfidf_cosine(s, t)
    jac   = _jaccard_keyword(s, t)
    ml    = ml_similarity(s, t)
    ml_scores.append(ml)
    print(f'[{i+1}] expected={l:6s}  tfidf={tfidf:.3f}  jaccard={jac:.3f}  → hybrid={ml:.3f}')

print(f'\nMean ML score: {np.mean(ml_scores):.3f}')

## 3. Model B — DL (Cross-encoder + SBERT bi-encoder)

In [ ]:
from answer_scorer import dl_similarity, _cross_encoder_score, _sbert_cosine

print('Cross-encoder + SBERT scores (Model B — DL)')
print('='*60)

dl_scores = []
for i, (t, s, l) in enumerate(SAMPLES):
    ce    = _cross_encoder_score(s, t)
    sbert = _sbert_cosine(s, t)
    dl    = dl_similarity(s, t)
    dl_scores.append(dl)
    print(f'[{i+1}] expected={l:6s}  cross-enc={ce:.3f}  sbert={sbert:.3f}  → combined={dl:.3f}')

print(f'\nMean DL score: {np.mean(dl_scores):.3f}')

## 4. Model C — Hybrid

In [ ]:
from answer_scorer import hybrid_similarity

print('Hybrid scores (Model C — Cross-enc + BiLSTM + TF-IDF)')
print('='*60)

hybrid_scores = []
for i, (t, s, l) in enumerate(SAMPLES):
    h = hybrid_similarity(s, t)
    hybrid_scores.append(h)
    print(f'[{i+1}] expected={l:6s}  → hybrid={h:.3f}')

print(f'\nMean Hybrid score: {np.mean(hybrid_scores):.3f}')

## 5. Comparison Bar Chart

In [ ]:
x      = np.arange(len(SAMPLES))
width  = 0.25
fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(x - width, ml_scores,     width, label='Model A — ML',     color='#f59e0b', alpha=.85)
ax.bar(x,         dl_scores,     width, label='Model B — DL',     color='#3b82f6', alpha=.85)
ax.bar(x + width, hybrid_scores, width, label='Model C — Hybrid', color='#10b981', alpha=.85)

ax.set_xticks(x)
ax.set_xticklabels([f'Pair {i+1}\n({l})' for i, (_, _, l) in enumerate(SAMPLES)], fontsize=10)
ax.set_ylabel('Similarity Score')
ax.set_ylim(0, 1.05)
ax.set_title('Model Comparison — Similarity Scores per Answer Pair')
ax.legend()
ax.axhline(0.7, color='gray', linestyle='--', linewidth=.8, alpha=.6)
ax.axhline(0.4, color='gray', linestyle=':', linewidth=.8, alpha=.6)
ax.text(len(SAMPLES)-.4, 0.71, 'Good threshold (0.7)', fontsize=9, color='gray')
ax.text(len(SAMPLES)-.4, 0.41, 'Below Avg threshold (0.4)', fontsize=9, color='gray')
plt.tight_layout()
plt.show()

## 6. BiLSTM Architecture Introspection

In [ ]:
from lstm_quality import BiLSTMScorer, NeuralQualityScorer
import torch

model_lstm = BiLSTMScorer()
total_params = sum(p.numel() for p in model_lstm.parameters())
trainable   = sum(p.numel() for p in model_lstm.parameters() if p.requires_grad)

print('BiLSTM Architecture:')
print(model_lstm)
print(f'\nTotal parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable:,}')

# Test forward pass with dummy sentence embeddings
dummy_input = torch.randn(2, 5, 768)  # batch=2, 5 sentences, 768-dim SBERT
with torch.no_grad():
    score, attn = model_lstm(dummy_input)

print(f'\nForward pass OK:')
print(f'  Input shape  : {dummy_input.shape}')
print(f'  Score shape  : {score.shape}')
print(f'  Attn shape   : {attn.shape}')
print(f'  Sample scores: {score.squeeze().tolist()}')

## 7. Regularization Analysis — Dropout Effect

In [ ]:
# Compare BiLSTM scores with different dropout rates to show regularization effect
dropout_rates = [0.0, 0.1, 0.3, 0.5]
variance_results = []

for dp in dropout_rates:
    m = BiLSTMScorer(dropout=dp)
    m.train()  # enable dropout
    scores_mc = []
    inp = torch.randn(1, 6, 768)
    with torch.no_grad():
        for _ in range(20):
            s, _ = m(inp)
            scores_mc.append(s.item())
    variance_results.append({'dropout': dp, 'mean': np.mean(scores_mc), 'std': np.std(scores_mc)})

print('Dropout rate vs prediction variance (Monte-Carlo, 20 samples):')
print(f'{"Dropout":>10}  {"Mean":>8}  {"Std Dev":>8}')
for r in variance_results:
    print(f"{r['dropout']:10.1f}  {r['mean']:8.4f}  {r['std']:8.4f}")

## Summary Table

In [ ]:
import pandas as pd

rows = []
for i, (t, s, l) in enumerate(SAMPLES):
    rows.append({
        'Pair': i+1,
        'Expected': l,
        'Model A (ML)': round(ml_scores[i], 3),
        'Model B (DL)': round(dl_scores[i], 3),
        'Model C (Hybrid)': round(hybrid_scores[i], 3),
    })

df = pd.DataFrame(rows)
df['Best Model'] = df[['Model A (ML)', 'Model B (DL)', 'Model C (Hybrid)']].idxmax(axis=1)
print(df.to_string(index=False))